[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leidyomuete/modelo-insolvencia-pymes/blob/main/Modelo_insolvencia_Colab.ipynb)

# Modelo predictivo para la identificación temprana del riesgo de insolvencia empresarial
### Empresas colombianas de los sectores manufacturero y comercial (Pymes, Grupo 2 NIIF, 2016–2025)

**Autora:** Leidy Briggitte Muete Garzón · **Tutora:** Gabriela María Saucedo Meza, Ph.D.
Maestría en Analítica de Datos para Contabilidad y Auditoría — Universidad Externado de Colombia

Este cuaderno ejecuta, paso a paso y sin instalar nada en el computador, el modelo descrito en el documento de investigación:
Random Forest complementado con SMOTE y contrastado con una regresión logística, con horizonte de predicción de tres años.

---
#### Cómo usarlo
1. Deje la fuente en **GitHub** (no requiere nada más) o elija **Google Drive** o **Google Drive** (la carpeta **Proyecto_Insolvencia**
   compartida por la autora debe aparecer en su Drive; si la recibió como enlace: clic derecho → *Organizar* → *Agregar acceso directo* → *Mi unidad*).
2. En el menú superior elija **Entorno de ejecución → Ejecutar todas** (o `Ctrl + F9`).
3. Si usa Google Drive, autorice el acceso cuando Colab lo solicite.

| Modo | Tiempo aproximado |
|---|---|
| Base consolidada + entrenamiento rápido | 8–12 minutos |
| Base consolidada + entrenamiento completo (como en la tesis) | 30–45 minutos |
| Excel originales del SIIS + entrenamiento completo | 45–60 minutos |

Los resultados se guardan en `resultados_colab`. El tablero interactivo con el simulador está publicado en
**https://leidyomuete.github.io/modelo-insolvencia-pymes/** (la celda 12b lo vuelve a generar si cambia el modelo).

In [ ]:
#@title 1. Configuración { display-mode: "form" }
#@markdown **Origen del código y de la base consolidada.** Con GitHub no se necesita Google Drive.
FUENTE = "GitHub"  #@param ["GitHub", "Google Drive"]
REPO_GITHUB = "https://github.com/leidyomuete/modelo-insolvencia-pymes"  #@param {type:"string"}
#@markdown Ruta de la carpeta del proyecto en Google Drive (solo si FUENTE = Google Drive o si usa los Excel originales):
CARPETA_PROYECTO = "/content/drive/MyDrive/Proyecto_Insolvencia"  #@param {type:"string"}
#@markdown Origen de los datos: la base consolidada (11 MB) reproduce exactamente la depuración de los 40 archivos Excel del SIIS.
MODO_DATOS = "Base consolidada (rápido)"  #@param ["Base consolidada (rápido)", "Excel SIIS originales (lento)"]
#@markdown Entrenamiento: el modo completo usa la misma búsqueda de hiperparámetros que la tesis (20 combinaciones, 5 pliegues).
MODO_ENTRENAMIENTO = "Rápido"  #@param ["Rápido", "Completo (como en la tesis)"]
#@markdown Definición de insolvencia (la del documento es la oficial; las otras son análisis de sensibilidad):
REGLA_ETIQUETA = "documento"  #@param ["documento", "estricta", "formal"]

In [ ]:
#@title 2. Obtener el código y verificar archivos
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
necesita_drive = FUENTE == "Google Drive" or not MODO_DATOS.startswith("Base")
if EN_COLAB and necesita_drive:
    from google.colab import drive
    drive.mount("/content/drive")

RAIZ = Path(CARPETA_PROYECTO)
BD = RAIZ / "bd"
if FUENTE == "GitHub":
    destino = Path("/content/repo_insolvencia") if EN_COLAB else Path("/tmp/repo_insolvencia")
    if not destino.exists():
        subprocess.run(["git", "clone", "-q", REPO_GITHUB.strip(), str(destino)], check=True)
    else:
        subprocess.run(["git", "-C", str(destino), "pull", "-q"], check=False)
    CODIGO = next(destino.rglob("main.py")).parent      # funciona si el código está en la raíz o en una subcarpeta
    # Si los archivos se subieron sin las carpetas src/ y data/, se reorganizan aquí
    for carpeta, archivos in {"src": ["__init__.py", "carga.py", "macro.py", "modelado.py", "reporte.py", "variables.py", "nuevas.py", "tablero.py", "tablero_plantilla.html"],
                              "data": ["macro_colombia.csv", "panel_consolidado.parquet", "plantilla_empresas_nuevas.xlsx"]}.items():
        (CODIGO / carpeta).mkdir(exist_ok=True)
        for a in archivos:
            if (CODIGO / a).exists() and not (CODIGO / carpeta / a).exists():
                (CODIGO / a).rename(CODIGO / carpeta / a)
    (CODIGO / "src" / "__init__.py").touch()
else:
    CODIGO = RAIZ / "modelo_insolvencia"

def verificar(ruta, descripcion):
    estado = "OK " if ruta.exists() else "FALTA"
    print(f"[{estado}] {descripcion}: {ruta}")
    return ruta.exists()

ok = verificar(CODIGO / "main.py", "Código del modelo")
ok &= verificar(CODIGO / "data" / "macro_colombia.csv", "Variables macroeconómicas")
if MODO_DATOS.startswith("Base"):
    ok &= verificar(CODIGO / "data" / "panel_consolidado.parquet", "Base consolidada")
else:
    ok &= verificar(BD / "pymes_individuales-2016", "Carpeta bd con los Excel del SIIS")
if not ok:
    raise FileNotFoundError("Revise la ruta CARPETA_PROYECTO del paso 1 o que la carpeta compartida esté en su Drive.")

In [ ]:
#@title 3. Instalar librerías (solo las que no trae Colab)
!pip install -q imbalanced-learn shap python-calamine

In [ ]:
#@title 4. Cargar el código y los parámetros metodológicos
import importlib, warnings
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

os.chdir(CODIGO)
if str(CODIGO) not in sys.path:
    sys.path.insert(0, str(CODIGO))
import config as cfg
importlib.reload(cfg)

cfg.RUTA_BD = BD
cfg.RUTA_RESULTADOS = CODIGO / "resultados_colab"
cfg.RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
cfg.REGLA_ETIQUETA = REGLA_ETIQUETA
if MODO_ENTRENAMIENTO == "Rápido":
    cfg.N_ITER_BUSQUEDA, cfg.N_FOLDS_CV, cfg.N_BOOTSTRAP = 5, 3, 300

from src import carga, variables, modelado, reporte, macro, nuevas, tablero
for m in (carga, variables, modelado, reporte, macro, nuevas, tablero):
    importlib.reload(m)
%matplotlib inline

pd.DataFrame({
    "Parámetro": ["Sectores (CIIU Rev. 4 A.C.)", "Años", "Horizonte de predicción", "Regla de etiqueta",
                  "Estados de insolvencia formal", "Partición", "Validación cruzada", "Combinaciones de hiperparámetros",
                  "Validación fuera de tiempo", "Niveles de riesgo"],
    "Valor": ["C (manufactura) y G (comercio)", f"{cfg.ANIOS[0]}–{cfg.ANIOS[-1]}", f"{cfg.HORIZONTE} años (t+1 a t+3)",
              cfg.REGLA_ETIQUETA, ", ".join(cfg.ESTADOS_INSOLVENCIA_FORMAL),
              f"{int((1-cfg.PROPORCION_PRUEBA)*100)} % / {int(cfg.PROPORCION_PRUEBA*100)} % por empresa",
              f"{cfg.N_FOLDS_CV} pliegues estratificados por empresa", cfg.N_ITER_BUSQUEDA,
              f"entrena ≤ {cfg.ANIO_CORTE_OOT}, prueba > {cfg.ANIO_CORTE_OOT}",
              "; ".join(f"{k}: {a:.0%}–{min(b,1):.0%}" for k, (a, b) in cfg.UMBRALES_RIESGO.items())],
}).style.hide(axis="index")

## Paso 1. Carga y depuración de los estados financieros
Se leen los reportes del SIIS de la Superintendencia de Sociedades (carátula, estado de situación financiera, estado de resultado integral y estado de flujos de efectivo),
se filtran las secciones CIIU C y G, los cortes al 31 de diciembre y los registros con ecuación patrimonial consistente.

In [ ]:
#@title 5. Construir el panel empresa-año
import time
t0 = time.time()
if MODO_DATOS.startswith("Base"):
    panel = pd.read_parquet(CODIGO / "data" / "panel_consolidado.parquet")
    print(f"Base consolidada cargada: {len(panel):,} estados financieros de {panel.nit.nunique():,} empresas")
else:
    macro.construir_macro()          # lee las series del Banco de la República de la carpeta bd
    panel = carga.construir_panel()  # lee los 40 archivos Excel del SIIS (~15 min)

resumen = panel.pivot_table(index="anio", columns="sector", values="nit", aggfunc="count", margins=True, margins_name="Total")
print(f"Tiempo: {(time.time()-t0)/60:.1f} min")
resumen

In [ ]:
#@title 6. Variables macroeconómicas utilizadas (Banco de la República / DANE)
pd.read_csv(cfg.RUTA_MACRO).set_index("anio")

## Paso 2. Indicadores financieros y variable dependiente
Se calculan los indicadores de la Tabla 1 (y las variables complementarias del Anexo 1) y se etiqueta cada observación del año *t*
como **en riesgo (y = 1)** si en alguno de los años *t+1, t+2 o t+3* cumple al menos un criterio de la definición operacional:
proceso formal (Ley 1116), patrimonio negativo, EBITDA negativo, razón corriente < 1 o cobertura de intereses < 1.

In [ ]:
#@title 7. Construir la base de modelado
import matplotlib.pyplot as plt
panel_ind, base = variables.construir_base_modelado(panel)
cols = variables.columnas_modelo()

crit = [c for c in panel_ind.columns if c.startswith("crit_")]
nombres = {"crit_ley1116": "Proceso formal (Ley 1116)", "crit_patrimonio_neg": "Patrimonio negativo",
           "crit_ebitda_neg": "EBITDA negativo", "crit_rc_menor1": "Razón corriente < 1",
           "crit_cobertura_menor1": "Cobertura de intereses < 1"}
prev = (panel_ind.groupby("sector")[crit].mean() * 100).rename(columns=nombres).T.round(1)
print("Frecuencia de cada criterio (% de estados financieros, 2016-2025):")
display(prev)

fig, ax = plt.subplots(figsize=(8, 3.5))
(base.groupby(["anio", "sector"]).y.mean().unstack() * 100).plot(ax=ax, marker="o")
ax.set_ylabel("% de empresas en riesgo"); ax.set_xlabel("Año de los estados financieros (t)")
ax.set_title("Proporción de empresas que presentan el evento en t+1 a t+3"); ax.grid(alpha=.3)
plt.show()
print(f"Observaciones para el modelo: {len(base):,}  |  variables explicativas: {len(cols)}")

## Paso 3. Entrenamiento y evaluación (70 % / 30 %)
Ambos modelos usan la misma secuencia: winsorización → imputación por la mediana → **SMOTE (solo en entrenamiento)** → modelo.
La partición se hace por empresa, de modo que una misma sociedad nunca aparece a la vez en entrenamiento y en prueba.

In [ ]:
#@title 8. Entrenar Random Forest + SMOTE y regresión logística + SMOTE
from IPython.display import Image, display
t0 = time.time()
rf, lr, metricas, importancia = modelado.evaluar_y_comparar(base, cols)
print(f"\nTiempo de entrenamiento: {(time.time()-t0)/60:.1f} min")

In [ ]:
#@title 9. Resultados en el conjunto de prueba
import json
comp = json.load(open(cfg.RUTA_RESULTADOS / "03_comparacion_modelos.json"))
tabla = metricas.drop(["VP", "FP", "FN", "VN"]).rename(index={
    "exactitud": "Exactitud", "precision": "Precisión", "sensibilidad_recall": "Sensibilidad (recall)",
    "especificidad": "Especificidad", "f1": "F1-score", "auc_roc": "AUC-ROC", "auc_pr": "AUC-PR", "brier": "Puntaje de Brier"})
display(tabla.round(3))
a, b = comp["IC95_diferencia"]
print(f"Diferencia de AUC (RF - Logit): IC 95 % = [{a:.3f}; {b:.3f}]  ->",
      "el Random Forest SUPERA a la regresión logística (cumple BRS-03)" if a > 0 else "no hay diferencia significativa")
display(Image(str(cfg.RUTA_RESULTADOS / "03_curva_roc.png"), width=480))

In [ ]:
#@title 10. Importancia de las variables
display(Image(str(cfg.RUTA_RESULTADOS / "03_importancia_variables.png"), width=620))
importancia.assign(variable=importancia.variable.map(lambda c: reporte.NOMBRES.get(c, c))).head(10).round(4)

In [ ]:
#@title 11. Desempeño por sector y validación fuera de tiempo
display(pd.read_csv(cfg.RUTA_RESULTADOS / "03_metricas_por_sector.csv", index_col=0).round(3))
oot = modelado.validacion_fuera_de_tiempo(base, cols, rf, lr)

## Paso 4. Calificación de empresas y reporte para el usuario
Se califican las empresas con estados financieros del último año disponible (2025): probabilidad de presentar insolvencia entre 2026 y 2028,
nivel de riesgo y los tres indicadores que más elevan su riesgo (valores SHAP).

In [ ]:
#@title 12. Generar el reporte
calificacion = reporte.calificar_empresas(panel_ind, rf, cols)
ruta_excel = reporte.exportar(calificacion, metricas, importancia)
reporte.guardar_modelo(rf, cols)

display(pd.crosstab(calificacion.sector, calificacion.nivel_riesgo, margins=True, margins_name="Total")
          [["Bajo", "Medio", "Alto", "Total"]])
print("\n20 empresas con mayor probabilidad de riesgo:")
calificacion.head(20)[["nit", "razon_social", "sector", "probabilidad_riesgo", "nivel_riesgo", "principales_factores"]]

In [ ]:
#@title 12b. Generar el tablero web (index.html) con los resultados y el simulador
ruta_tablero = tablero.generar_tablero(panel_ind, base, cols, rf, lr, cfg.RUTA_RESULTADOS / "index.html")
print("Súbalo a la raíz del repositorio de GitHub para actualizar https://leidyomuete.github.io/modelo-insolvencia-pymes/")
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_tablero))

In [ ]:
#@title 13. Consultar una empresa por NIT (calificación actual y trayectoria de riesgo)
NIT = 0  #@param {type:"integer"}
if NIT:
    r = calificacion[calificacion.nit == NIT]
    if r.empty:
        print("El NIT no está en la base calificada (verifique que sea manufacturera o comercial y que reporte en 2025).")
        print("Si es una empresa nueva, use la celda 15 o 16 con sus estados financieros.")
    else:
        display(r.T)
        hist = panel_ind[panel_ind.nit == NIT].sort_values("anio").copy()
        hist["probabilidad"] = rf.predict_proba(hist[cols])[:, 1]
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.plot(hist.anio, hist.probabilidad * 100, marker="o", color="#333")
        for nivel, (a, b) in cfg.UMBRALES_RIESGO.items():
            ax.axhspan(a * 100, min(b, 1) * 100, alpha=.08, color={"Bajo": "green", "Medio": "orange", "Alto": "red"}[nivel])
        ax.set_ylim(0, 100); ax.set_ylabel("% probabilidad de riesgo"); ax.set_xlabel("Año de los estados financieros")
        ax.set_title(f"Trayectoria de riesgo — {r.razon_social.iloc[0]}"); ax.grid(alpha=.3); plt.show()
        print("Nota: los años anteriores a 2023 se usaron para entrenar el modelo; la trayectoria es descriptiva.")
else:
    print("Escriba un NIT en el campo de la derecha y vuelva a ejecutar esta celda.")

In [ ]:
#@title 14. Descargar el reporte en Excel
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_excel))
else:
    print("Reporte en:", ruta_excel)

## Paso 5. Calificar empresas que no están en la base de Supersociedades
Para una empresa nueva (por ejemplo, un cliente de auditoría que no reporta al SIIS) el modelo necesita las cifras de sus estados financieros,
**en miles de pesos**. Use la celda 15 para una empresa o la celda 16 para varias empresas con la plantilla de Excel.
El resultado es la probabilidad de presentar señales de insolvencia en los tres años siguientes al cierre de esos estados financieros.

In [ ]:
#@title 15. Calificar una empresa nueva (escriba las cifras en miles de pesos)
RAZON_SOCIAL = "EMPRESA EJEMPLO SAS"  #@param {type:"string"}
NIT_NUEVA = 900123456  #@param {type:"integer"}
SECTOR = "Manufacturero"  #@param ["Manufacturero", "Comercial"]
ANIO_EEFF = 2025  #@param {type:"integer"}
FECHA_CONSTITUCION = "2012-03-15"  #@param {type:"string"}
OPINION_REVISOR = "Limpio"  #@param ["Limpio", "Con salvedad", "Negativa", "Abstención"]
#@markdown **Estado de situación financiera**
ACTIVO_CORRIENTE = 4500000  #@param {type:"number"}
ACTIVO_TOTAL = 9800000  #@param {type:"number"}
PASIVO_CORRIENTE = 3900000  #@param {type:"number"}
PASIVO_TOTAL = 6200000  #@param {type:"number"}
PATRIMONIO = 3600000  #@param {type:"number"}
GANANCIAS_ACUMULADAS = 1100000  #@param {type:"number"}
PRESTAMOS_CORTO_PLAZO = 1200000  #@param {type:"number"}
PRESTAMOS_LARGO_PLAZO = 1500000  #@param {type:"number"}
#@markdown **Estado de resultados y flujo de efectivo**
INGRESOS = 12400000  #@param {type:"number"}
INGRESOS_ANIO_ANTERIOR = 13100000  #@param {type:"number"}
UTILIDAD_OPERACIONAL = 310000  #@param {type:"number"}
GASTOS_FINANCIEROS = 420000  #@param {type:"number"}
UTILIDAD_NETA = -95000  #@param {type:"number"}
DEPRECIACION_AMORTIZACION = 280000  #@param {type:"number"}
FLUJO_OPERATIVO = 150000  #@param {type:"number"}

empresa = pd.DataFrame([{
    "NIT": NIT_NUEVA, "Razón social": RAZON_SOCIAL, "Sector (Manufacturero/Comercial)": SECTOR,
    "Año de los estados financieros": ANIO_EEFF, "Fecha de constitución": FECHA_CONSTITUCION,
    "Opinión del revisor fiscal": OPINION_REVISOR, "Activo corriente": ACTIVO_CORRIENTE, "Activo total": ACTIVO_TOTAL,
    "Pasivo corriente": PASIVO_CORRIENTE, "Pasivo total": PASIVO_TOTAL, "Patrimonio": PATRIMONIO,
    "Ganancias acumuladas": GANANCIAS_ACUMULADAS, "Préstamos corto plazo": PRESTAMOS_CORTO_PLAZO,
    "Préstamos largo plazo": PRESTAMOS_LARGO_PLAZO, "Ingresos operacionales": INGRESOS,
    "Ingresos del año anterior": INGRESOS_ANIO_ANTERIOR, "Utilidad operacional": UTILIDAD_OPERACIONAL,
    "Gastos financieros": GASTOS_FINANCIEROS, "Utilidad neta": UTILIDAD_NETA,
    "Depreciación y amortización": DEPRECIACION_AMORTIZACION, "Flujo de efectivo de operación": FLUJO_OPERATIVO}])
res = nuevas.calificar_nuevas(empresa, rf, cols).iloc[0]
print(f"{res.razon_social}: probabilidad de riesgo {res.probabilidad_riesgo:.1%} -> nivel {res.nivel_riesgo.upper()} "
      f"(para {res['riesgo_para_los_años']})")
print("Indicadores que más elevan el riesgo:", res.principales_factores)
print("Señales de alerta actuales:", res.alertas_actuales)
pd.DataFrame(res).rename(columns={res.name: "Resultado"})

In [ ]:
#@title 16. Calificar varias empresas con la plantilla de Excel
#@markdown 1) Ejecute esta celda con `ACCION = Descargar plantilla`, llénela (una fila por empresa) y guárdela.
#@markdown 2) Cambie a `ACCION = Subir y calificar`, ejecute de nuevo y seleccione el archivo.
ACCION = "Descargar plantilla"  #@param ["Descargar plantilla", "Subir y calificar"]
plantilla = CODIGO / "data" / "plantilla_empresas_nuevas.xlsx"
if not plantilla.exists():
    nuevas.crear_plantilla(plantilla)
if ACCION == "Descargar plantilla":
    if EN_COLAB:
        from google.colab import files; files.download(str(plantilla))
    else:
        print("Plantilla en:", plantilla)
else:
    if EN_COLAB:
        from google.colab import files
        subido = files.upload()
        archivo = next(iter(subido))
    else:
        archivo = str(plantilla)
    lote = pd.read_excel(archivo, sheet_name="Empresas")
    resultado = nuevas.calificar_nuevas(lote, rf, cols)
    salida = cfg.RUTA_RESULTADOS / "calificacion_empresas_nuevas.xlsx"
    resultado.to_excel(salida, index=False)
    display(resultado)
    if EN_COLAB:
        files.download(str(salida))

---
#### Notas de reproducibilidad
* Semilla aleatoria fija (`42`) en la partición, SMOTE y los modelos.
* En el modo **rápido** la búsqueda de hiperparámetros es más corta; las métricas pueden variar en el tercer decimal frente al documento.
  Para reproducir exactamente la tesis use *Completo (como en la tesis)*.
* Todas las decisiones metodológicas están en `config.py`.
* Con FUENTE = GitHub los resultados quedan en la sesión de Colab (se pierden al cerrarla); descargue el Excel con la celda 14.
* Fuentes: Superintendencia de Sociedades — SIIS (estados financieros NIIF Pymes individuales 2016–2025); Banco de la República y DANE (tasa de política monetaria, inflación, PIB y TRM).